# Dataset Generation (Documentation Only)

 **DO NOT RUN THIS NOTEBOOK UNLESS YOU WANT TO RECREATE THE DATASET FROM SCRATCH** 

The full dataset generated by this script is already hosted and available for download on Kaggle: 
https://www.kaggle.com/datasets/viditarora18/speech-sep-mixtures/

This notebook is provided purely for transparency, showing how the LibriMix recipe was extended to handle 4 and 5 concurrent speakers using the `make_mixtures.py` script.

In [ ]:
%%writefile make_mixtures.py
"""
make_mixtures.py

Generates N-speaker mixtures from LibriSpeech-style data, extending the
LibriMix recipe (which natively only goes up to 3 speakers) to 4, 5, or
more concurrent speakers.

For each mixture it writes:
  - mix.wav              the summed mixture
  - s1.wav, s2.wav, ...   the individual clean sources (ground truth,
                          in the SAME order they were added -- this order
                          is what you'll use as the OR-PIT extraction
                          target order during training)
  - a row in metadata.csv describing the mixture

Usage:
    python make_mixtures.py \
        --librispeech-root /path/to/LibriSpeech/train-clean-100 \
        --out-dir /path/to/output/mix4 \
        --num-speakers 4 \
        --num-mixtures 2000 \
        --sample-rate 16000 \
        --min-mode

Directory expectation for --librispeech-root: standard LibriSpeech layout,
i.e. <root>/<speaker_id>/<chapter_id>/<speaker_id>-<chapter_id>-<utt_id>.flac
(or .wav -- both are auto-detected).

Design notes (why it's built this way):
  - "min mode" (default, like LibriMix's min mode) trims all sources to the
    length of the SHORTEST source in the mixture before summing, so every
    sample in mix.wav has all N speakers active -- this is the harder,
    more honest version of the task and matches how LibriMix's min mode
    is normally used for training separation models.
  - Loudness is randomized per-source (uniform dB offset) before mixing,
    the same way LibriMix does it, so the model doesn't learn to rely on
    "the loudest voice is always speaker 1" as a shortcut it can't count on
    at test time.
  - Speakers within one mixture are sampled WITHOUT replacement (no two
    utterances from the same speaker in one mixture) -- this matters more
    than it sounds like it should, because otherwise the "how many
    speakers" signal becomes ambiguous even to a human listener.
"""

import argparse
import csv
import random
from pathlib import Path

import numpy as np
import soundfile as sf


AUDIO_EXTS = (".flac", ".wav")


def find_speaker_utterances(librispeech_root: Path) -> dict:
    """Map speaker_id -> list of utterance file paths."""
    speakers = {}
    for spk_dir in sorted(librispeech_root.iterdir()):
        if not spk_dir.is_dir():
            continue
        spk_id = spk_dir.name
        utts = [p for p in spk_dir.rglob("*") if p.suffix.lower() in AUDIO_EXTS]
        if utts:
            speakers[spk_id] = utts
    return speakers


def load_audio(path: Path, sample_rate: int) -> np.ndarray:
    audio, sr = sf.read(str(path), dtype="float32")
    if audio.ndim > 1:  # collapse to mono if needed
        audio = audio.mean(axis=1)
    if sr != sample_rate:
        raise ValueError(
            f"{path} is {sr} Hz, expected {sample_rate} Hz. "
            f"Resample your source data first (e.g. with torchaudio.transforms.Resample) "
            f"-- resampling on the fly here would silently vary quality per file."
        )
    return audio


def rms_normalize(audio: np.ndarray, target_dbfs: float) -> np.ndarray:
    """Scale audio so its RMS level matches target_dbfs (dB relative to full scale)."""
    rms = np.sqrt(np.mean(audio**2)) + 1e-9
    target_rms = 10 ** (target_dbfs / 20)
    return audio * (target_rms / rms)


def make_one_mixture(
    speaker_ids: list,
    speakers: dict,
    sample_rate: int,
    min_mode: bool,
    loudness_range_db: tuple,
    rng: random.Random,
) -> tuple:
    """Returns (mixture, [source_arrays], [chosen_file_paths])."""
    sources = []
    chosen_paths = []
    for spk_id in speaker_ids:
        utt_path = rng.choice(speakers[spk_id])
        audio = load_audio(utt_path, sample_rate)
        gain_db = rng.uniform(*loudness_range_db)
        audio = rms_normalize(audio, gain_db)
        sources.append(audio)
        chosen_paths.append(utt_path)

    if min_mode:
        length = min(len(s) for s in sources)
        sources = [s[:length] for s in sources]
    else:
        length = max(len(s) for s in sources)
        sources = [np.pad(s, (0, length - len(s))) for s in sources]

    mixture = np.sum(sources, axis=0)

    # Peak-normalize the mixture (and sources by the same factor) to avoid
    # clipping when several loud sources stack -- keeps everything in [-1, 1].
    peak = np.max(np.abs(mixture)) + 1e-9
    if peak > 0.99:
        scale = 0.99 / peak
        mixture = mixture * scale
        sources = [s * scale for s in sources]

    return mixture, sources, chosen_paths


def generate_split(
    librispeech_root: Path,
    out_dir: Path,
    num_speakers: int,
    num_mixtures: int,
    sample_rate: int,
    min_mode: bool,
    loudness_range_db: tuple,
    seed: int,
):
    rng = random.Random(seed)
    speakers = find_speaker_utterances(librispeech_root)
    speaker_ids_all = list(speakers.keys())

    if len(speaker_ids_all) < num_speakers:
        raise ValueError(
            f"Only found {len(speaker_ids_all)} speakers under {librispeech_root}, "
            f"need at least {num_speakers}."
        )

    out_dir.mkdir(parents=True, exist_ok=True)
    metadata_path = out_dir / "metadata.csv"

    with open(metadata_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(
            ["mixture_id", "num_speakers"]
            + [f"speaker_{i+1}_id" for i in range(num_speakers)]
            + [f"speaker_{i+1}_source_file" for i in range(num_speakers)]
        )

        for i in range(num_mixtures):
            mix_id = f"mix_{i:06d}"
            speaker_ids = rng.sample(speaker_ids_all, num_speakers)

            mixture, sources, chosen_paths = make_one_mixture(
                speaker_ids, speakers, sample_rate, min_mode, loudness_range_db, rng
            )

            mix_folder = out_dir / mix_id
            mix_folder.mkdir(exist_ok=True)
            sf.write(str(mix_folder / "mix.wav"), mixture, sample_rate)
            for j, src in enumerate(sources):
                sf.write(str(mix_folder / f"s{j+1}.wav"), src, sample_rate)

            writer.writerow(
                [mix_id, num_speakers]
                + speaker_ids
                + [str(p) for p in chosen_paths]
            )

            if (i + 1) % 200 == 0:
                print(f"  {i + 1}/{num_mixtures} mixtures written")

    print(f"Done. {num_mixtures} mixtures -> {out_dir}")
    print(f"Metadata -> {metadata_path}")


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--librispeech-root", type=Path, required=True,
                     help="Path to a LibriSpeech split, e.g. train-clean-100")
    ap.add_argument("--out-dir", type=Path, required=True)
    ap.add_argument("--num-speakers", type=int, required=True, choices=range(2, 10))
    ap.add_argument("--num-mixtures", type=int, required=True)
    ap.add_argument("--sample-rate", type=int, default=16000)
    ap.add_argument("--min-mode", action="store_true", default=True,
                     help="Trim to shortest source (default, matches LibriMix min mode)")
    ap.add_argument("--max-mode", dest="min_mode", action="store_false",
                     help="Pad to longest source instead")
    ap.add_argument("--loudness-min-db", type=float, default=-33.0)
    ap.add_argument("--loudness-max-db", type=float, default=-25.0)
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    generate_split(
        librispeech_root=args.librispeech_root,
        out_dir=args.out_dir,
        num_speakers=args.num_speakers,
        num_mixtures=args.num_mixtures,
        sample_rate=args.sample_rate,
        min_mode=args.min_mode,
        loudness_range_db=(args.loudness_min_db, args.loudness_max_db),
        seed=args.seed,
    )


if __name__ == "__main__":
    main()

In [ ]:
# Example of how the training splits were generated:
# !python make_mixtures.py \
#   --librispeech-root ../data/LibriSpeech/train-clean-100 \
#   --out-dir ../data/train_mix2 \
#   --num-speakers 2 --num-mixtures 3000 --seed 42

# (Repeat commented-out commands for 3, 4, 5 speakers and validation sets...)

In [ ]:
!zip -r /kaggle/working/mixtures.zip /kaggle/working/train_mix* /kaggle/working/val_mix*